# Synthetic tRBS cases: a benchmark with a known answer

## In one minute

When we optimise a real tRBS case, we get an allocation and a score. What we
never get is the answer to the obvious question: **was that actually the best
allocation possible?** On a real case nobody knows the true optimum, so there is
nothing to compare against.

This notebook shows the way around that. We *build* tRBS cases whose best
allocation we work out in advance, then feed them to the optimiser and measure
exactly how far off it was.

- **The problem:** on real cases, "did the optimiser find the best plan?" cannot
  be answered.
- **What this does:** generates realistic tRBS cases where the best plan is
  known beforehand.
- **What you get:** an exact error number (the *gap*) for any optimisation
  method, at any problem size.

The generated cases are ordinary tRBS cases. They are written as the same tables
a real case uses and run through the same pipeline, so the optimiser cannot tell
them apart from a real one.

## 1. Why this matters

The main contribution of this thesis is a **convexity characterisation** of tRBS
cases: a rule that says when the appreciation landscape is convex, and whether
that predicts if an optimiser finds the true best allocation.

A useful picture:

- **Convex** is a single mountain. Walk uphill from wherever you start and you
  always end up at the same summit, which is the highest point.
- **Non-convex** is a mountain range. Walk uphill and you reach the nearest
  peak, which is not necessarily the highest one. Where you started decides
  where you end up.

On real cases such as Beerwiser, Refugee and IZZ we cannot check which situation
we are in, because the true summit is unknown. Claiming "the optimiser found the
best plan" would be an act of faith. Generated cases replace that faith with a
measurement.

## 2. How it works

The generator writes a case as the 11 standard tRBS tables (`key_outputs`,
`decision_makers_options`, `dependencies`, `scenarios` and so on). tRBS reads
those tables from CSV, Excel or JSON; here we write CSV. Because the tables are
the real thing, the case flows through the normal
`build` to `evaluate` to `appreciate` to `optimize` pipeline untouched.

Alongside the tables the generator writes a `manifest.json` holding the settings
that produced the case, its convexity claim, and later the certified best
allocation computed by the **oracle**.

Each case is described by three settings:

- **`regime`**: the shape of the landscape. `convex` (one mountain),
  `smooth_nonconvex` (several rounded peaks) or `nonsmooth` (peaks with sharp
  ridges and flat plateaus).
- **`k`**: the number of internal variables, meaning the things we can spend the
  budget on. This is the size of the problem.
- **`seed`**: the starting value of the random number generator. Everything
  random in a case (the coefficients, the weights) is drawn from it, so the same
  seed always rebuilds exactly the same case, file for file. It is what makes
  the results reproducible by anyone.

For every combination of these three the oracle computes the best possible
allocation. The question we can then finally ask is: does the optimiser find it?

## 3. Who this is for

Anyone who wants to know how much to trust an optimisation result on a tRBS
case, and anyone who wants to test a new optimisation method against a known
answer before trusting it on real client work. It assumes you have seen a tRBS
case before; if not, `vlinder_demo.ipynb` in the repository root walks through
the basics first.

## 4. Setup

Two things to know before running this notebook.

Run it in the repository's virtual environment, the one where `vlinder` is
installed in editable mode, and not against a plain `pip install vlinder`. The
notebook needs both the installed `vlinder` package and the thesis modules
`case_factory` and `oracle`, which are not shipped with the released package.

The working directory must be `experiments/synthetic/`, which it is, since the
notebook lives there. That is what makes the plain `import case_factory` work
and puts generated cases in `experiments/synthetic/generated/`.

In [ ]:
import pandas as pd

import case_factory as cf
import oracle
from study_harness import StudyHarness, StudySpec

from vlinder.trbs import TheResponsibleBusinessSimulator
from vlinder.optimize import Optimize

# Note: vlinder.__init__ exposes only TheResponsibleBusinessSimulator and
# list_demo_cases, so Optimize is imported from vlinder.optimize directly.

## 5. Generate a simple case

We start with the most straightforward example: a convex case with three
internal variables and three KPIs.

The parameters we set:

- `name`: the case name, used for the output folder.
- `k=3`: three internal variables, so three things to divide the budget over.
- `n_key_outputs=3`: three KPIs to score the result on.
- `seed=1`: fixes the random draws, so this exact case can be rebuilt later.

Everything else stays at its default: `regime='convex'`,
`appreciation='linear'`, and `budget=100.0`, meaning any allocation has to
satisfy sum(x) <= 100.

The parameters that would introduce non-convexity are all switched off, which
the convex regime requires: `n_stb1=0` (no KPIs flipped to smaller-is-better),
`n_bilinear=0` (no multiplicative couplings between variables),
`n_saturation=0` (no capped variables) and `bracketing_factor=1.0` (no
clipping). Section 8 turns these on one at a time.

In [ ]:
params = cf.SyntheticCaseParams(name="Synthetic_convex_k3", k=3, n_key_outputs=3, seed=1)
params

### 5.1 Write it to disk

`.write()` produces the 11 tRBS tables under `<root>/<name>/csv/` plus the
`manifest.json`. Running it twice with the same parameters is safe: it writes
exactly the same bytes.

In [ ]:
root = cf.SyntheticCaseFactory(params).write()
print(f"Case written to: {root}")

### 5.2 Look at what came out

These are ordinary tRBS tables. Below are the decision makers options, which set
the range each internal variable can move in.

In [ ]:
cf.SyntheticCaseFactory(params).tables()["decision_makers_options"]

## 6. Run it through the real pipeline

Nothing here is specific to synthetic cases. This is the same sequence as
`vlinder_demo.ipynb`: point the simulator at the tables and run the pipeline.

In [ ]:
sim = TheResponsibleBusinessSimulator(params.name, file_path=root, file_extension="csv")
sim.build()
sim.evaluate()
sim.appreciate()
print("Pipeline complete: build, evaluate, appreciate")

### 6.1 Optimise with multi-start SLSQP

Now the part we are actually studying. SLSQP is a gradient-based solver: from a
starting allocation it walks uphill until it cannot improve. Because it stops at
the first peak it reaches, we restart it from 20 random points and keep the best
result.

The arguments:

- `scenario`: which scenario to optimise for.
- `budget`: the spending cap, passed explicitly rather than inferred.
- `dmo_name="SLSQP"`: the name the winning allocation is stored under. Naming it
  after the method keeps results apart when several methods run on one case.
- `n_starts=20`: how many random starting points to try.
- `seed=1`: makes those starting points reproducible.

In [ ]:
scenario = str(sim.input_dict["scenarios"][0])
print(f"Optimising scenario: {scenario}")

opt = Optimize(sim.input_dict, sim.output_dict)
slsqp = opt.optimize_slsqp(scenario, params.budget, dmo_name="SLSQP", n_starts=20, seed=1)

print(f"SLSQP appreciation: {slsqp.appreciation:.4f}")
print(f"Allocation (x):     {slsqp.allocation}")
print(f"Converged:          {slsqp.n_converged}/{slsqp.n_starts} starts")

## 7. Certify the answer with the oracle

`certify_case()` reads the regime from the manifest, picks the matching oracle,
computes the best possible allocation and writes it back into `manifest.json`.

For this case the appreciation is linear, so the score is proportional to how
much we put into each variable. The optimum then has to sit at a corner of the
feasible region. In plain terms: when everything is proportional, the best plan
is to put the entire budget into the single most rewarding variable. So it is
enough to check the three "all budget on one variable" plans plus the "spend
nothing" plan, and take the winner. The oracle does exactly that, and separately
computes the gradient by hand to confirm it points at the same corner.

In [ ]:
oracle_result = oracle.certify_case(params.name, root)

print(f"Oracle method: {oracle_result['method']}")
print(f"Verified:      {oracle_result['verified']}")
print(f"Oracle f* for {scenario}: {oracle_result['per_scenario'][scenario]['f_star']:.4f}")

### 7.1 What the oracle can and cannot guarantee

Worth being precise about, because the answer differs per case type. "Certain"
below means mathematically proven, not "we checked a lot of points".

| Case type | How the oracle finds the answer | How certain is it |
|---|---|---|
| convex, linear | Checks the k "all budget on one variable" plans plus "spend nothing" | **Certain**, at any size k |
| convex, curved | Tight local solver plus a KKT check | **Certain**: on a convex problem, a local optimum is the global one |
| non-convex, k <= 6 | Dense grid over the whole budget space, then polishing the best points | **Certain at grid resolution**: a peak narrower than the grid spacing could be missed |
| non-convex, k > 6 | Best of many local solves from different starting points | **Not certain**: this is a best effort, not a proof |

So for a non-convex case the oracle does **not** know the global optimum in
general. That is a real limit, not a detail. Two things follow from it, and both
are deliberate:

1. Above k = 6 the certificate is labelled `verified: false`. Numbers measured
   against it compare methods with each other, and are never read as "method X
   found the true optimum".
2. The thesis anchors its claim about absolute optimum recovery on the **convex**
   regime only, where the answer is provable at any problem size. For non-convex
   cases we report evidence about the shape of the landscape (how many peaks it
   has) rather than a recovery claim.

### 7.2 Compare: did SLSQP find it?

The payoff. The **gap** is the distance between the certified best score and
what the optimiser actually achieved: `gap = f_star - f_found`. A gap of zero
means the optimiser found the best allocation. On a real case this number cannot
be computed at all.

In [ ]:
f_star = oracle_result["per_scenario"][scenario]["f_star"]
gap = f_star - slsqp.appreciation

print(f"f* (oracle): {f_star:.4f}")
print(f"f  (SLSQP):  {slsqp.appreciation:.4f}")
print(f"Gap:         {gap:.2e}")
print()
print("SLSQP recovered the certified optimum." if gap < 1e-6 else f"SLSQP missed it by {gap:.4f}.")

### 7.3 Checking the case itself

Before trusting any gap we have to trust the case. `validate_case()` re-imports
it and checks a list of properties, raising an error if any fails:

- every evaluation produces a real number, no NaN or infinity;
- every KPI value stays inside the **envelope**, meaning the lowest and highest
  value that KPI can reach anywhere within the budget. The envelope is worked out
  algebraically beforehand, so this checks the maths against the actual pipeline;
- **no clipping**: tRBS scores a KPI on a 0 to 100 scale between the lowest and
  highest value it observes. If those boundaries sit too close together, values
  get cut off at 0 or 100, which puts artificial creases in the landscape. This
  check confirms the boundaries are wide enough that no cutting off happens;
- the allocation SLSQP returns respects the budget;
- for convex cases, all 20 starts reached the same score, which is what a single
  mountain implies.

It returns a dictionary of diagnostics rather than a bare pass or fail.

In [ ]:
diagnostics = cf.validate_case(params.name, root, budget=params.budget)

print("Validation diagnostics:")
for key in ["no_clip", "slsqp_consensus", "consensus_spread", "n_distinct_terminal_values"]:
    print(f"  {key}: {diagnostics[key]}")

## 8. Cases with more than one peak

Convex cases are the easy baseline. The interesting question is what happens
when the landscape has several peaks, so that where the optimiser starts decides
where it finishes.

### 8.1 Smooth peaks

The `smooth_nonconvex` regime bends the landscape without creating sharp edges,
through two parameters:

- `n_stb1`: how many KPIs are flipped to smaller-is-better and scored on a curve.
  This is the IZZ cost-KPI mechanism: spending more starts to hurt.
- `n_bilinear`: how many KPIs get a term multiplying two internal variables
  together, so their effects depend on each other rather than simply adding up.

The boundaries stay wide enough that no clipping occurs, so every peak here
comes from genuine curvature and not from an artefact.

### 8.2 Generate and certify one

Because k <= 6, the oracle uses the dense grid method from the table in section
7.1: it evaluates a fine grid covering the whole budget space, then polishes the
best points with a local solver. It also estimates the number of **basins**. A
basin is one valley in the landscape, in the sense that a ball released anywhere
inside it rolls to the same bottom. More basins means more separate peaks, and
more ways for an optimiser to get stuck on the wrong one.

In [ ]:
params2 = cf.SyntheticCaseParams(
    name="Synthetic_smooth_k3",
    k=3,
    n_key_outputs=3,
    regime="smooth_nonconvex",
    appreciation="sinusoidal",
    n_stb1=1,
    n_bilinear=1,
    seed=4,
)

root2 = cf.SyntheticCaseFactory(params2).write()
oracle2 = oracle.certify_case(params2.name, root2)

print(f"Oracle method: {oracle2['method']}")
print(f"Verified:      {oracle2['verified']}")

### 8.3 Read the basin count

`n_basins_f_estimate` and `n_basins_x_estimate` count the distinct end points
the local solver reached, by score and by allocation.

Notice the result: at this small size, curvature alone still leaves **one**
peak. That is worth stating plainly, because it is easy to assume that
"non-convex" automatically means "full of traps". It does not. Whether a bent
landscape actually acquires a second peak depends on how strongly it is bent and
on how many variables there are, which is exactly the thing this apparatus
measures instead of assuming. Section 8.4 shows a mechanism that does reliably
produce a second peak at this size.

In [ ]:
sim2, _ = cf.build_case(params2.name, root2, "Probe")
scenario2 = str(sim2.input_dict["scenarios"][0])
cert = oracle2["per_scenario"][scenario2]["certificate"]

print(f"Grid resolution:             {cert.get('grid_resolution', 'n/a')}")
print(f"Estimated basins (by score): {cert.get('n_basins_f_estimate', 'n/a')}")
print(f"Estimated basins (by plan):  {cert.get('n_basins_x_estimate', 'n/a')}")
print()

diag2 = cf.validate_case(params2.name, root2, budget=params2.budget)
print(f"No clipping still holds:  {diag2['no_clip']}")
print(f"Distinct end values:      {diag2['n_distinct_terminal_values']}")

### 8.4 Sharp edges, and a second peak

There is a second route to multiple peaks, and on this size of problem it is the
one that actually works. The `nonsmooth` regime deliberately narrows the KPI
boundaries using `bracketing_factor`, a dial between 0 and 1.

At `1.0` the boundaries exactly span what the KPIs can reach, so nothing is cut
off. Below `1.0` they close in, and KPI values start hitting the 0 and 100 ends
of the scale inside the feasible region. Everything past that point scores
identically, which creates flat plateaus with sharp creases at their edges.

This is not a cosmetic difference. Cutting off at the bottom of the scale is
what splits a single peak into two, and it was originally found as a **bug** in
an earlier version of this generator: cases labelled convex were quietly
multi-peaked, so any optimiser measured against them would have been scored
against the wrong answer. It is now kept as a deliberate setting so the effect
can be studied rather than suffered.

The validator recognises the narrowing and confirms it is intentional instead of
raising an error.

In [ ]:
params3 = cf.SyntheticCaseParams(
    name="Synthetic_nonsmooth_clip_k3",
    k=3,
    n_key_outputs=3,
    regime="nonsmooth",
    bracketing_factor=0.6,
    seed=3,
)

root3 = cf.SyntheticCaseFactory(params3).write()
diag3 = cf.validate_case(params3.name, root3, budget=params3.budget)
oracle3 = oracle.certify_case(params3.name, root3)
cert3 = next(iter(oracle3["per_scenario"].values()))["certificate"]

print(f"No clipping holds:       {diag3['no_clip']}")
print(f"Narrowing margin:        {diag3.get('underbracketing_margin', 'n/a')}")
print(f"Share of points clipped: {diag3.get('clipping_incidence', 'n/a')}")
print()
print(f"Estimated basins (by score): {cert3.get('n_basins_f_estimate', 'n/a')}")
print(f"Estimated basins (by plan):  {cert3.get('n_basins_x_estimate', 'n/a')}")
print()
print("The validator confirms this narrowing is intentional, so it does not raise an error.")

## 9. From one case to the scaling claim

A single case is an anecdote. The headline claim of the thesis is about
**scale**: as the number of internal variables k grows, does the optimiser still
find the true best allocation in reasonable time? Two tools support that.

### 9.1 The standard suite

Ten cases covering every regime and every mechanism at least once. This is the
regression set: if a change to the generator breaks something, it shows up here.

In [ ]:
suite = cf.standard_cases()
coverage = [(p.name, p.regime, p.k, p.appreciation) for p in suite]
print(f"Standard suite ({len(suite)} cases):")
print(pd.DataFrame(coverage, columns=["name", "regime", "k", "appreciation"]).to_string())

### 9.2 The study grid

The full pre-registered study runs both convex variants over
k = 2, 3, 4, 6, 9, 12, 15 with 30 seeds each and four optimisation methods,
which is 5,040 individual runs. `StudyHarness` handles it: it generates each
case, certifies it, runs every method and records one row per result, resuming
where it left off if interrupted.

Below is a small version of the same thing, so it finishes while you watch.
The full design is fixed in `PREREGISTRATION.md`.

One thing to expect: the harness prints how many tasks it still has to run. The
first time you run this cell that number is 18. Run it again and it prints 0,
because every result is already on disk and it does not recompute them. That is
the same mechanism that let the real 5,040-run study survive being interrupted.

In [ ]:
spec = StudySpec(
    variants=("linear", "sinusoidal"),
    ks=(2, 3, 4),
    seeds=(0,),
    methods=("slsqp",),
    root=str(cf.DEFAULT_ROOT / "notebook_demo"),
)
harness = StudyHarness(spec)
results_path = harness.run(n_workers=1)

rows = pd.read_json(results_path, lines=True)
print()
print(rows[["case_name", "variant", "k", "method", "f_oracle", "gap", "recovered"]].to_string(index=False))

### 9.3 What that shows

Across every k tested, the gap on the convex-linear variant stays at the level
of numerical rounding error. That is the checkable version of the claim "the
method scales": not "it looked fine", but "it recovered the provably best
allocation at every size we tried".

The scope limit from section 7.1 applies here too. This absolute claim rests on
the convex regime, because that is where the answer is provable. For non-convex
cases we report how rugged the landscape is and verify at grid resolution for
small k, and we do not claim the global optimum was recovered.

## 10. Extending the generator

To add a new regime or mechanism, work through these in order.

1. **`SyntheticCaseParams` and `__post_init__`** in `case_factory.py`: add the
   setting as a field, and add its rules to the validation matrix, for example
   "this setting is only valid in regime X". That matrix is the single source of
   truth for which combinations are allowed.
2. **`SyntheticCaseFactory`** in `case_factory.py`: draw any new randomness from
   a *new* `SeedSequence` child stream. Never reuse an existing one, because
   that would make an unrelated setting change the draws. Then build the
   mechanism into the relevant table methods, keeping the boundaries wide enough
   that nothing clips, unless clipping is the point of your mechanism.
3. **`envelope()` and `manifest()`** in `case_factory.py`: extend the envelope
   if your mechanism changes which KPI values are reachable, and update the
   convexity claim.
4. **`Oracle.certify`** in `oracle.py`: decide which oracle certifies the new
   regime, and be explicit about whether it proves the answer or estimates it.
5. **`validate_case`** in `case_factory.py`: add the invariant that should hold
   for your regime.
6. **`test_case_factory.py`**: add your case to `test_knob_validation_matrix` so
   invalid combinations are rejected, and add a round-trip test.

### The reproducibility contract

Two tests protect it. `test_determinism_byte_identical` requires that the same
parameters and seed produce byte-identical files. `test_subseed_isolation`
requires that changing one setting does not re-randomise anything unrelated.
Any new mechanism has to keep both passing.

## 11. Where to look next

- [`case_factory.py`](case_factory.py): builds the cases.
- [`oracle.py`](oracle.py): computes the certified answers.
- [`study_harness.py`](study_harness.py): runs the full study grid.
- [`test_case_factory.py`](test_case_factory.py): worked examples and regression
  tests.
- [`PREREGISTRATION.md`](PREREGISTRATION.md): the frozen study design.
- [`vlinder_demo.ipynb`](../../vlinder_demo.ipynb): the basics of tRBS itself.